In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")

In [2]:
users = pd.read_csv(r"C:\Users\kuldiep\Downloads\Data\users_clean.csv")
events = pd.read_csv(r"C:\Users\kuldiep\Downloads\Data\events_clean.csv")
orders = pd.read_csv(r"C:\Users\kuldiep\Downloads\Data\orders_clean.csv")

In [3]:
users["signup_date"] = pd.to_datetime(users["signup_date"])
events["timestamp"] = pd.to_datetime(events["timestamp"])
orders["order_date"] = pd.to_datetime(orders["order_date"])

In [4]:
funnel_flags = events.pivot_table(
    index = "user_id",
    columns = "event",
    values = "timestamp",
    aggfunc = "min"
).reset_index()

funnel_flags.columns.name = None

In [7]:
funnel_flags = funnel_flags.rename(columns={
    "visit": "visit_time",
    "signup": "signup_time",
    "activate": "activate_time",
    "add_to_cart": "add_to_cart_time",
    "purchase": "purchase_time"
})

In [8]:
for col in ["visit_time", "signup_time", "activate_time", "add_to_cart_time", "purchase_time"]:
    flag_name = col.replace("_time", "_flag")
    funnel_flags[flag_name] = np.where(funnel_flags[col].notna(), 1, 0)

In [9]:
order_summary = orders.groupby("user_id").agg(
    total_orders=("order_id", "count"),
    total_revenue=("order_value", "sum"),
    avg_order_value=("order_value", "mean"),
    first_order_date=("order_date", "min")
).reset_index()

In [10]:
funnel = users.merge(funnel_flags, on="user_id", how="left")
funnel = funnel.merge(order_summary, on="user_id", how="left")

In [11]:
flag_cols = ["visit_flag", "signup_flag", "activate_flag", "add_to_cart_flag", "purchase_flag"]
for col in flag_cols:
    funnel[col] = funnel[col].fillna(0).astype(int)

funnel["total_orders"] = funnel["total_orders"].fillna(0).astype(int)
funnel["total_revenue"] = funnel["total_revenue"].fillna(0)
funnel["avg_order_value"] = funnel["avg_order_value"].fillna(0)

In [12]:
funnel["visit_time"] = pd.to_datetime(funnel["visit_time"])
funnel["signup_time"] = pd.to_datetime(funnel["signup_time"])
funnel["activate_time"] = pd.to_datetime(funnel["activate_time"])
funnel["add_to_cart_time"] = pd.to_datetime(funnel["add_to_cart_time"])
funnel["purchase_time"] = pd.to_datetime(funnel["purchase_time"])
funnel["first_order_date"] = pd.to_datetime(funnel["first_order_date"])

funnel["days_visit_to_signup"] = (funnel["signup_time"] - funnel["visit_time"]).dt.days
funnel["days_signup_to_activate"] = (funnel["activate_time"] - funnel["signup_time"]).dt.days
funnel["days_activate_to_cart"] = (funnel["add_to_cart_time"] - funnel["activate_time"]).dt.days
funnel["days_cart_to_purchase"] = (funnel["purchase_time"] - funnel["add_to_cart_time"]).dt.days
funnel["days_visit_to_purchase"] = (funnel["purchase_time"] - funnel["visit_time"]).dt.days

In [14]:
funnel["converted_user"] = np.where(funnel["purchase_flag"] == 1, 1, 0)

In [15]:
def stage_reached(row):
    if row["purchase_flag"] == 1:
        return "Purchase"
    elif row["add_to_cart_flag"] == 1:
        return "Add to Cart"
    elif row["activate_flag"] == 1:
        return "Activate"
    elif row["signup_flag"] == 1:
        return "Signup"
    elif row["visit_flag"] == 1:
        return "Visit"
    return "Unknown"

funnel["max_stage_reached"] = funnel.apply(stage_reached, axis=1)

In [16]:
funnel["revenue_segment"] = pd.cut(
    funnel["total_revenue"],
    bins=[-1, 0, 500, 1500, 3000, np.inf],
    labels=["No Revenue", "Low", "Medium", "High", "Very High"]
)

In [17]:
funnel.to_csv("funnel_user_summary.csv", index=False)